# W8 — chạy **TEST MỘT LẦN** với cấu hình đã khóa (freeze v2), Kaggle T4

Cấu hình khóa 11/11: **P(tau 0.20, M = 5) + F** (`results/countgdpp/freeze_v2_20261111.json`, sha256 `64e89ef5…`). Notebook kiểm hash dữ liệu test và freeze, chạy runner **một lần** (B0 = lượt 1, B2 và tau0.200 trong cùng run), chấm CLIP (`p1_controls.py`), gom bundle, rồi chấm **duy nhất** bằng `apply_freeze_v2.py`.

**Cài đặt:** giống W6, gồm **GPU T4 x2**, **Internet On**, gắn dataset checkpoint (Add Input). Chạy bằng **Save Version → Save & Run All (Commit)**.

**Quy tắc (GIAO_VIEC mục W8):**
- Không chọn lại ngưỡng, không chạy chấm nhanh `evaluate_research.py`, không chạy lại vì kết quả xấu.
- Chỉ chạy lại đúng lệnh cũ khi có **lỗi kỹ thuật** (crash, thiếu file), và ghi chú vào WORKLOG.
- Môi trường (Ô 3) và Kiểm A (Ô 5) chép nguyên từ notebook W6.

| Ô | Làm gì |
|---|---|
| 1–2 | GPU T4; lấy code; kiểm `test/inputs.jsonl`, `test/manifest.csv` khớp hash đã khóa, freeze v2 đúng sha256 |
| 3–5 | Môi trường, dữ liệu (kiểm hash 40 ảnh test), checkpoint, Kiểm A (giống W6) |
| 6 | **Chạy test một lần:** tau 0.20, M = 5, `--replay-b2` (120 query) |
| 7 | `p1_controls.py` → `bundle_offline_v2.py` → **`apply_freeze_v2.py`** (kết quả chính + đối chứng phụ, bootstrap theo ảnh) |
| 8 | Gói `for_repo_<stamp>.zip` (file nhỏ, D10) + raw trong Output |

In [ ]:
# Ô 1 — thiết lập. KHÔNG đổi tham số chạy: lấy từ freeze v2 ở Ô 2.
import datetime, glob, hashlib, json, os, pathlib, re, shlex, shutil, subprocess, sys, zipfile
REPO_URL = "https://github.com/minhtridaoo/zero-shot-counting.git"
REPO_REF = "main"
FREEZE_SHA = "64e89ef53eab306fdbb33cfc39797749ff17cc8635f46ae381d96e45c4f6736d"  # GIAO_VIEC mục W8
STAMP = datetime.datetime.now(datetime.timezone(datetime.timedelta(hours=7))).strftime("%Y%m%d_%H%M")
RUN_ID = f"p1_test_kaggle_t4_{STAMP}"
ROOT = pathlib.Path("/tmp/CS331")
PY = "/tmp/venv310/bin/python"
OUT = pathlib.Path("/kaggle/working/w8_out")
assert not any(pathlib.Path("/kaggle/working").glob("w8_out/p1_test_*")), "Phiên này đã chạy test rồi. Dừng: test chỉ chạy một lần."
OUT.mkdir(parents=True, exist_ok=True)
ENV = os.environ.copy()
ENV.update(CUDA_VISIBLE_DEVICES="0", MPLBACKEND="Agg", PYTHONUNBUFFERED="1")
def sh(cmd, cwd=None):
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait():
        raise RuntimeError(f"Lệnh lỗi (mã {p.returncode}), xem các dòng ngay phía trên. Dừng, không chạy tiếp.")
def file_hash(path, algo="sha256"):
    h = hashlib.new(algo)
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()
gpu = subprocess.check_output("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader", shell=True, text=True)
print(gpu)
assert "T4" in gpu.splitlines()[0], "Cần GPU T4 (Session options > Accelerator > GPU T4 x2). Dừng."
print("run id:", RUN_ID)

In [ ]:
# Ô 2 — lấy code; kiểm dữ liệu test đã khóa và cấu hình đã khóa (freeze v2).
if not ROOT.exists():
    sh(f"git clone -q {REPO_URL} {ROOT}")
sh(f"git fetch -q origin && git checkout -q {REPO_REF} && (git pull -q --ff-only 2>/dev/null || true)", cwd=ROOT)
COMMIT = subprocess.check_output("git rev-parse HEAD", shell=True, cwd=ROOT, text=True).strip()
print("commit", COMMIT)
SET = ROOT / "experiments/countgdpp/dev_test_v2"
meta = json.loads((SET / "selection_meta.json").read_text(encoding="utf-8"))
assert meta["status"] == "LOCKED", "dev_test_v2 chưa LOCKED. Dừng."
for f in ("test/inputs.jsonl", "test/manifest.csv"):
    assert file_hash(SET / f) == meta["locked_file_sha256"][f], f"{f} khác hash đã khóa. Dừng."
FREEZE = ROOT / "results/countgdpp/freeze_v2_20261111.json"
assert file_hash(FREEZE) == FREEZE_SHA, "freeze v2 khác sha256 đã ghi trong GIAO_VIEC. Dừng."
fz = json.loads(FREEZE.read_text(encoding="utf-8"))
assert fz["status"] == "FROZEN_V2"
TAU = " ".join(f"{t:.2f}" for t in fz["runner"]["args"]["tau_pseudo"])
M = fz["runner"]["args"]["m"]
assert fz["runner"]["args"]["replay_b2"] is True and TAU == "0.20" and M == 5, (TAU, M)
TEST_INPUTS = "experiments/countgdpp/dev_test_v2/test/inputs.jsonl"
TEST_MANIFEST = "experiments/countgdpp/dev_test_v2/test/manifest.csv"
n = sum(1 for _ in open(ROOT / TEST_INPUTS, encoding="utf-8"))
assert n == 120, n
print(f"test LOCKED {meta['locked_at']} | {n} query | hash test khớp | freeze v2 khớp: {fz['main']['name']} (tau {TAU}, M {M})")

In [ ]:
# Ô 3 — môi trường /tmp/venv310 giống Colab đã qua Kiểm A/C (torch 2.5.1+cu121, transformers 4.44.2, numpy 2.2.6, pillow 10.4.0)
# và build CUDA ops (MultiScaleDeformableAttention) cho T4 (sm 7.5). Dùng nvcc 12.x có sẵn của Kaggle nếu có, không thì tải CUDA 12.1.
sh("pip install -q uv")
if not os.path.exists(PY):
    sh("uv venv /tmp/venv310 --python 3.10")
sh(f"uv pip install -q -p {PY} torch==2.5.1 torchvision==0.20.1 --index-url https://download.pytorch.org/whl/cu121")
sh(f"uv pip install -q -p {PY} transformers==4.44.2 timm==1.0.30 numpy==2.2.6 pillow==10.4.0 scipy termcolor addict yapf==0.40.1"
   " opencv-python-headless pycocotools matplotlib setuptools wheel ninja")
nvccs = sorted(set(filter(None, [shutil.which("nvcc"), *glob.glob("/usr/local/cuda*/bin/nvcc")])))
cuda_home = None
for nv in nvccs:
    if re.search(r"release 12\.", subprocess.run([nv, "--version"], capture_output=True, text=True).stdout):
        cuda_home = str(pathlib.Path(nv).resolve().parent.parent); break
if cuda_home is None:
    cuda_home = "/tmp/cuda-12.1"
    if not os.path.exists(f"{cuda_home}/bin/nvcc"):
        sh("wget -q -O /tmp/cuda121.run https://developer.download.nvidia.com/compute/cuda/12.1.1/local_installers/cuda_12.1.1_530.30.02_linux.run")
        sh(f"sh /tmp/cuda121.run --silent --toolkit --toolkitpath={cuda_home} --override --no-man-page && rm -f /tmp/cuda121.run")
sh(f"{cuda_home}/bin/nvcc --version | tail -1")
sh("apt-get -qq update && apt-get -qq install -y gcc-11 g++-11 > /dev/null")
ENV.update(CUDA_HOME=cuda_home, PATH=f"{cuda_home}/bin:/tmp/venv310/bin:" + ENV["PATH"], CC="gcc-11", CXX="g++-11", TORCH_CUDA_ARCH_LIST="7.5")
sh("rm -rf /tmp/ops_build /tmp/wheels && cp -r " + str(ROOT / "CountGDPlusPlus/models/GroundingDINO/ops") + " /tmp/ops_build")
sh("rm -rf /tmp/ops_build/build /tmp/ops_build/dist /tmp/ops_build/*.egg-info")
sh(f"cd /tmp/ops_build && {PY} setup.py -q bdist_wheel -d /tmp/wheels")
wheel = sorted(p for p in glob.glob("/tmp/wheels/*.whl") if os.path.basename(p).lower().startswith("multiscaledeformableattention-"))[-1]
sh(f"uv pip install -q -p {PY} --reinstall {wheel}")
probe = ("import sys, torch, transformers, numpy, PIL, MultiScaleDeformableAttention; "
         "assert torch.__version__.startswith('2.5.1'), torch.__version__; "
         "print('python', sys.version.split()[0], '| torch', torch.__version__, '| transformers', transformers.__version__, "
         "'| numpy', numpy.__version__, '| pillow', PIL.__version__, '| GPU', torch.cuda.get_device_name())")
sh(f"{PY} -c {shlex.quote(probe)}")
sh(f"uv pip freeze -p {PY} > {OUT}/packages.txt")

In [ ]:
# Ô 4 — ảnh MUCCA 1.0 (Zenodo, kiểm md5), checkpoint CountGD++ (kiểm sha256 đã audit 14/10), BERT.
# Nếu gdown báo quá lượt tải: tạo Kaggle Dataset riêng tư chứa countgd_plusplus.pth (bản trên Drive MyDrive/CS331_colab/),
# gắn vào notebook (Add Input) rồi chạy lại; ô này tự tìm file trong /kaggle/input.
MUCCA = ROOT / "data_external/mucca_zenodo_19231375/MUCCA_1.0"
if not (MUCCA / "images/mcac_61.jpg").exists():
    sh('wget -q -O /tmp/MUCCA_1.0.zip "https://zenodo.org/api/records/19231375/files/MUCCA_1.0.zip/content"')
    md5 = file_hash("/tmp/MUCCA_1.0.zip", "md5")
    assert md5 == "b182e445445dad017493ffb35803f816", f"md5 sai: {md5}"
    MUCCA.mkdir(parents=True, exist_ok=True)
    sh(f"unzip -q -o /tmp/MUCCA_1.0.zip -d {MUCCA}")
print("Số ảnh MUCCA:", len(list((MUCCA / "images").glob("*.jpg"))), "(phải là 200)")
for img, digest in meta["image_sha256"].items():
    if img in meta["test"]:
        assert file_hash(MUCCA / f"images/{img}.jpg") == digest, f"ảnh {img} khác hash đã khóa"
print("40 ảnh test khớp hash đã khóa")
CKPT = ROOT / "CountGDPlusPlus/checkpoints/countgd_plusplus.pth"
EXPECTED = json.loads((ROOT / "results/countgdpp/checkpoint_audit_2026-10-14.json").read_text())["checkpoint_sha256"]
if not CKPT.exists():
    local = glob.glob("/kaggle/input/**/countgd_plusplus.pth", recursive=True)
    if local:
        shutil.copy(local[0], CKPT)
    else:
        sh(f"uv pip install -q -p {PY} gdown")
        sh(f'/tmp/venv310/bin/gdown "https://drive.google.com/uc?id=1j6N22TtKu2NVcKpgfrf-sJHGeLDqs9hs" -O {CKPT}')
assert file_hash(CKPT) == EXPECTED, "checkpoint sha256 sai"
print("Checkpoint đúng bản đã audit 14/10.")
BERT = ROOT / "CountGDPlusPlus/checkpoints/bert-base-uncased"
if not any((BERT / n).exists() for n in ("model.safetensors", "pytorch_model.bin")):
    sh(f"{PY} download_bert.py", cwd=ROOT / "CountGDPlusPlus")
print("BERT:", sorted(p.name for p in BERT.iterdir()))

In [ ]:
# Ô 5 — KIỂM A: CLIP trên Kaggle cho điểm giống bản đã lưu (chấm lại khung cũ của p4_unseen_v1, không chỉnh gì).
check = """
import json, sys
from pathlib import Path
from PIL import Image
ROOT = Path("/tmp/CS331"); sys.path.insert(0, str(ROOT / "experiments/countgdpp"))
from pseudo_clip_verifier import ClipVerifier
clip = ClipVerifier("cuda")
diffs = []
for f in sorted((ROOT / "results/countgdpp/p4_unseen_v1_clip_20261027").glob("*_clip_scores.json")):
    r = json.loads(f.read_text(encoding="utf-8"))
    rows = r["rows_original_order"]
    if not rows:
        continue
    with Image.open(ROOT / r["image_path"]) as im:
        image = im.convert("RGB")
    got = clip.score(image, [x["original_box_xyxy"] for x in rows], r["prompt"])
    diffs += [abs(a - x["clip_cosine_target"]) for a, x in zip(got, rows)]
print(f"KIEM A: {len(diffs)} khung, lech lon nhat {max(diffs):.2e}, trung binh {sum(diffs)/len(diffs):.2e}")
assert max(diffs) < 1e-3, "Kiem A KHONG DAT"
"""
pathlib.Path("/tmp/clip_check.py").write_text(check, encoding="utf-8")
sh(f"{PY} /tmp/clip_check.py", cwd=ROOT)

In [ ]:
# Ô 6 — CHẠY TEST MỘT LẦN: tau 0.20, M = 5, kèm B2 gốc từ cùng lượt 1 (B0 = lượt 1). Model không đọc GT.
sh(f"{PY} experiments/countgdpp/run_b2_verified_pseudo.py --inputs {TEST_INPUTS} --run-id {RUN_ID}"
   f" --tau-pseudo {TAU} --m {M} --replay-b2 --check-inputs", cwd=ROOT)  # chỉ kiểm input/checkpoint/code
sh(f"{PY} experiments/countgdpp/run_b2_verified_pseudo.py --inputs {TEST_INPUTS} --run-id {RUN_ID}"
   f" --tau-pseudo {TAU} --m {M} --replay-b2", cwd=ROOT)
cfg = json.loads((ROOT / "results/countgdpp" / RUN_ID / "run_config.json").read_text(encoding="utf-8"))
assert cfg["status"] == "complete", cfg["status"]
shutil.copytree(ROOT / "results/countgdpp" / RUN_ID, OUT / RUN_ID)
print("Xong", RUN_ID, "| outputs", cfg["outputs"], "| GPU", cfg.get("device"))

In [ ]:
# Ô 7 — CHẤM DUY NHẤT bằng cấu hình đã khóa: điểm CLIP (p1_controls) -> bundle -> apply_freeze_v2. Không chọn lại ngưỡng.
CTRL = RUN_ID + "_controls"
sh(f"{PY} experiments/countgdpp/p1_controls.py --run results/countgdpp/{RUN_ID} --run-id {CTRL}", cwd=ROOT)
sh(f"{PY} experiments/countgdpp/bundle_offline_v2.py --controls results/countgdpp/{CTRL}"
   f" --out results/countgdpp/{RUN_ID}_offline/bundle.json", cwd=ROOT)
sh(f"{PY} experiments/countgdpp/apply_freeze_v2.py --bundle results/countgdpp/{RUN_ID}_offline/bundle.json"
   f" --manifest {TEST_MANIFEST} --out results/countgdpp/{RUN_ID}_eval", cwd=ROOT)
for d in (CTRL + "_clip", CTRL, RUN_ID + "_offline", RUN_ID + "_eval"):
    shutil.copytree(ROOT / "results/countgdpp" / d, OUT / d)
res = json.loads((OUT / (RUN_ID + "_eval") / "result.json").read_text(encoding="utf-8"))
print("\nKẾT QUẢ TEST (", res["status"], ") | loại uncertain:", res["uncertain_excluded"])

In [ ]:
# Ô 8 — GÓI KẾT QUẢ: for_repo_<stamp>.zip chỉ gồm json/jsonl/csv/txt (D10, KHÔNG .npz, KHÔNG ảnh); raw đủ trong w8_out/.
sh(f"uv pip freeze -p {PY} > {OUT}/packages.txt")
allowed = {".json", ".jsonl", ".csv", ".txt"}
zip_path = pathlib.Path(f"/kaggle/working/for_repo_{STAMP}.zip")
listing = []
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.rglob("*")):
        if p.is_file():
            listing.append(dict(path=str(p.relative_to(OUT)), bytes=p.stat().st_size, sha256=file_hash(p)))
            if p.suffix in allowed:
                z.write(p, p.relative_to(OUT))
(OUT / f"FILES_{STAMP}.json").write_text(json.dumps(dict(commit=COMMIT, stamp=STAMP, run_id=RUN_ID, freeze_sha256=FREEZE_SHA, files=listing), indent=2), encoding="utf-8")
with zipfile.ZipFile(zip_path, "a") as z:
    z.write(OUT / f"FILES_{STAMP}.json", f"FILES_{STAMP}.json")
raw = sum(1 for x in listing if x["path"].endswith(".npz"))
print(f"{zip_path.name}: {round(zip_path.stat().st_size / 1e6, 1)} MB | w8_out: {len(listing)} file, {raw} raw .npz | commit {COMMIT[:7]}")
print("XONG W8. Tải for_repo_*.zip gửi assistant. KHÔNG chạy lại notebook này vì kết quả.")

## Sau khi chạy xong

1. Tab **Output** của version → tải **`for_repo_<stamp>.zip`** về `Downloads` và báo assistant. Assistant kiểm hash rồi chép file D10 cho phép vào `results/countgdpp/`.
2. Gửi phần in ra của **Ô 2**, **Ô 5** và **Ô 7** (bảng kết quả).
3. Kết quả tốt hay xấu đều **không chạy lại**. Chỉ khi lỗi kỹ thuật mới chạy lại đúng notebook này, và ghi chú vào WORKLOG.